<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/02_demand_eda_single_product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 02 · Demand EDA — Single Product

**Data Science · Università Cattolica del Sacro Cuore, Milan**
Instructor: *Paolo Balasso*

---

## What you will explore

A complete EDA + baseline-forecasting workflow on a **single SKU** daily demand series, building the foundation before any complex forecasting model.

1. Generate a realistic synthetic daily sales series (trend + weekly + yearly + noise + outliers) — and check the description against the data
2. **Decomposition** — trend / seasonality / residual (additive STL, robust vs non-robust)
3. **Stationarity** — Augmented Dickey-Fuller test (light touch)
4. **Autocorrelation** — ACF / PACF
5. **Outlier handling** — rolling IQR + a known-event calendar, masking, replacement strategies
6. **Baseline forecasts** — Naive, Seasonal-Naive, Linear trend, Holt-Winters (ETS)
7. **Accuracy metrics** — MAPE / WAPE / MASE + bias (slide recap + interpretation)
8. **Residual diagnostics** — what does the model fail to capture?

**Style.** Short readable code, lots of plots. Mini-exercises are *parameter tweaks* — change seasonality, change holdout, observe.

---

## 0 · Setup

In [ ]:
# Colab: every library below is preinstalled, nothing to install.
# Local Jupyter: uncomment the next line and run it once.
# %pip install -q "statsmodels>=0.14" matplotlib seaborn pandas numpy scipy

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)   # hide library deprecation chatter only
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

NAVY, GOLD, INK, MUTED = '#1B263B', '#D4A017', '#0F141E', '#556070'
GREEN, RED = '#2D6E3E', '#B32D2D'
sns.set_style('whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK})
np.random.seed(42)
print(f'Setup complete. numpy {np.__version__} · pandas {pd.__version__} · '
      f'statsmodels {statsmodels.__version__}')

## 1 · A realistic synthetic SKU

We build a daily-frequency series that mimics a real sales feed:
- a positive **trend**;
- a **weekly cycle**: sales peak Monday–Wednesday and dip on Friday–Saturday (think of a product bought mostly on working days);
- a **yearly cycle**: a summer peak around July, then a decline through Q4 into January;
- Gaussian **noise**;
- two planted outliers: a 5-day **promo spike** (28 Oct – 1 Nov 2022) and a 3-day **stockout dip** (26–28 Apr 2023).

> Never trust a description of the data — this one included. The cell after the plot checks the weekly and yearly pattern against the numbers.

In [ ]:
def make_sku(start='2022-01-01', n_days=730, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start=start, periods=n_days, freq='D')
    t = np.arange(n_days)

    trend     = 100 + 0.06*t                                      # +6 units/100 days
    weekly    = 12*np.sin(2*np.pi*(t % 7)/7 - 1.0)                # peaks Mon-Wed, trough Fri-Sat
    yearly    = 25*np.sin(2*np.pi*(t % 365)/365 - 1.8)            # summer peak (July), declines through Q4
    noise     = rng.normal(0, 6, n_days)
    series    = trend + weekly + yearly + noise

    # Plant a promo spike (5 days, 28 Oct-1 Nov 2022) and a stockout dip (3 days, 26-28 Apr 2023)
    promo_start, promo_len = 300, 5
    series[promo_start:promo_start+promo_len] += 60               # spike
    stockout_start, stockout_len = 480, 3
    series[stockout_start:stockout_start+stockout_len] *= 0.10    # dip

    return pd.Series(series.clip(min=0), index=dates, name='units_sold')

y = make_sku()
print(f'SKU series: {len(y)} days, from {y.index.min().date()} to {y.index.max().date()}')
print(f'Mean: {y.mean():.1f}, Std: {y.std():.1f}, Min: {y.min():.1f}, Max: {y.max():.1f}')
y.head()

In [ ]:
# Eyeball: full series + zoom on the planted anomalies
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(y.index, y.values, color=NAVY, lw=0.9)
axes[0].set_title('Daily units sold (full sample)',
                   fontweight='bold', color=INK)
axes[0].axvspan(y.index[300], y.index[305], color=GREEN, alpha=0.25, label='Promo')
axes[0].axvspan(y.index[480], y.index[483], color=RED,   alpha=0.25, label='Stockout')
axes[0].legend(loc='upper left')

# Zoom on October-December: weekly cycle on top of the falling yearly wave
zoom = y.loc['2022-10-01':'2022-12-31']
axes[1].plot(zoom.index, zoom.values, color=NAVY, lw=1.1)
axes[1].set_title('Zoom: Q4 2022 (weekly cycle on a falling yearly wave; promo at end of October)',
                   fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

In [ ]:
# Check the story against the data: average units by weekday and by month
by_wday = y.groupby(y.index.dayofweek).mean().round(1)
by_wday.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
by_month = y.groupby(y.index.month).mean().round(1)
by_month.index = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print('Average units by weekday:')
print(by_wday.to_frame('units').T.to_string())
print('\nAverage units by month (2022 and 2023 together):')
print(by_month.to_frame('units').T.to_string())
print(f'\nPeak weekday: {by_wday.idxmax()}  ·  Peak month: {by_month.idxmax()}  ·  '
      f'Lowest month: {by_month.idxmin()}')

## 2 · STL decomposition — separating signal from noise

STL = Seasonal-Trend decomposition using LOESS. Robust, handles outliers, lets us see each component independently.

With `period=7`, STL models **only the weekly cycle**. The slow yearly wave has nowhere else to go, so it ends up in the **Trend** panel: here "trend" means *everything slower than a week*, not linear growth. Modelling both cycles at once (e.g. `MSTL` with periods 7 and 365) is the natural next step.

In [ ]:
# Weekly seasonality, since the data are daily
PERIOD = 7                       # mini-exercise: try 5, 30, then 14
stl = STL(y, period=PERIOD, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
parts = [
    ('Observed',   y,             NAVY),
    ('Trend',      stl.trend,     GOLD),
    ('Seasonal',   stl.seasonal,  GREEN),
    ('Residual',   stl.resid,     RED),
]
for ax, (label, series, color) in zip(axes, parts):
    ax.plot(series.index, series.values, color=color, lw=0.9)
    ax.set_title(label, fontweight='bold', color=INK, fontsize=10, loc='left')
plt.suptitle(f'STL decomposition (period = {PERIOD} days)',
             fontweight='bold', color=INK, y=1.005)
plt.tight_layout(); plt.show()
planted = y.index[300:305].union(y.index[480:483])        # the 8 promo + stockout days
print(f'Residual std, all days:                  {stl.resid.std():.2f}')
print(f'Residual std, without the 8 planted days: {stl.resid.drop(planted).std():.2f}')
print('True noise in the generator (we know it): 6.00')

**Read-out — the noise floor.** The generator adds noise with standard deviation 6: that is the **irreducible noise**, the floor no model can go below on average. The STL residual (about 10.3) is larger because robust STL deliberately leaves the promo and stockout in the residual. Drop those 8 days and it falls to about 4.9 — *below* 6, because the flexible trend and seasonal curves also soak up part of the noise. So the residual std is a rough guide to the noise floor, not an exact measurement. On real data you never know the true noise: treat the residual as a ballpark.

In [ ]:
# Same decomposition WITHOUT robust mode: where does the promo go?
stl_nr = STL(y, period=PERIOD, robust=False).fit()   # same period as the STL cell above
win = y.loc['2022-10-01':'2022-11-30']

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(win.index, win.values, color=MUTED, lw=0.8, label='Observed')
ax.plot(win.index, stl.trend[win.index], color=GOLD, lw=2.2, label='Trend, robust=True')
ax.plot(win.index, stl_nr.trend[win.index], color=RED, lw=2.0, ls='--', label='Trend, robust=False')
ax.axvspan(y.index[300], y.index[305], color=GREEN, alpha=0.2, label='Promo')
ax.set_title('Robust vs non-robust STL around the promo',
             fontweight='bold', color=INK)
ax.legend(loc='upper left', fontsize=8); plt.tight_layout(); plt.show()

promo = y.index[300:305]
print(f'Average trend on the promo days: robust {stl.trend[promo].mean():.1f}  vs  '
      f'non-robust {stl_nr.trend[promo].mean():.1f}')
print(f'Average residual on the promo days: robust {stl.resid[promo].mean():.1f}  vs  '
      f'non-robust {stl_nr.resid[promo].mean():.1f}')

> **Mini-exercise (parametric).** In the STL cell, change `PERIOD = 7` to `PERIOD = 5`, then to `PERIOD = 30`, and re-run. These are *not* multiples of the true 7-day cycle: the seasonal component shrinks and the weekly structure leaks into the residual (watch the residual std go up). Then try `PERIOD = 14`: almost nothing changes, because 14 is a multiple of 7 and still contains the weekly cycle. STL is *not* finding the period for you — you have to know it. Set `PERIOD` back to 7 when you are done.

## 3 · Stationarity check (light touch)

Many forecasting methods assume the series is **stationary** (constant mean and variance). The **Augmented Dickey-Fuller (ADF)** test: H0 = "the series has a unit root" (non-stationary). Small p-value → reject → stationary.

In [ ]:
def adf_summary(series, label):
    stat, p, *_ = adfuller(series.dropna(), autolag='AIC')
    verdict = 'STATIONARY (reject H0)' if p < 0.05 else 'NON-stationary'
    print(f'{label:30s}  ADF stat={stat:7.3f}  p={p:.4f}   → {verdict}')

adf_summary(y, 'Raw series')
adf_summary(y.diff().dropna(), '1st differences')
adf_summary(stl.resid, 'STL residual')

**Read-out.** The raw series is non-stationary (p ≈ 0.23: trend plus the yearly wave). First differences and the STL residual are clearly stationary (p < 0.001). This shapes which model class is appropriate: ARIMA differences the series first (the *d* in ARIMA(p, d, q)), while ETS / Holt-Winters handles trend natively.

## 4 · Autocorrelation — ACF & PACF

- **ACF**: correlation of the series with its own lags. Tells you how far in the past the signal still echoes.
- **PACF**: same but *partialing out* shorter lags. Diagnostic for AR-order selection (self-study).

On a series with a trend, the raw ACF stays high at every lag and decays slowly: the weekly bumps at lags 7, 14, 21 are there, but they sit on top of the trend. Remove the trend first (first differences, `y.diff()`) and the weekly spikes at lags 7, 14, 21 stand out clearly.

In [ ]:
dy = y.diff().dropna()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_acf(y, lags=30, ax=axes[0], color=NAVY)
axes[0].set_title('ACF — raw daily sales', fontweight='bold', color=INK)
plot_acf(dy, lags=30, ax=axes[1], color=NAVY)
axes[1].set_title('ACF — first differences (trend removed)', fontweight='bold', color=INK)
plot_pacf(y, lags=30, ax=axes[2], method='ywm', color=NAVY)
axes[2].set_title('PACF — raw (self-study)', fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

r_raw, r_d = acf(y, nlags=21), acf(dy, nlags=21)
print(f'Raw ACF, lags 1 / 7 / 14:          {r_raw[1]:.2f} / {r_raw[7]:.2f} / {r_raw[14]:.2f}'
      '  → slow decay = trend not yet removed')
print(f'Differenced ACF, lags 1 / 7 / 14 / 21: {r_d[1]:.2f} / {r_d[7]:.2f} / {r_d[14]:.2f} / {r_d[21]:.2f}'
      '  → spikes every 7 days = weekly seasonality')

## 5 · Outlier detection & treatment

Outliers fall into two business categories:
- **Real events to remember** (promo, holiday) — keep them but flag them, so a later model can use the flag
- **Data errors** (stockout, scanner glitch) — replace with a robust estimate. A stockout is special: sales collapse because the shelf was empty, not because nobody wanted the product. The real demand on those days was never observed.

We combine two sources:
- a **rolling-window IQR rule** — a transparent statistical detector (`is_out`);
- a **known-event calendar** — the promo dates marketing tells you it ran (`is_promo`).

Watch what the detector finds on its own, and what it misses.

In [ ]:
def detect_outliers_iqr(series, window=14, k=2.5):
    rolling = series.rolling(window, center=True, min_periods=5)
    q1 = rolling.quantile(0.25); q3 = rolling.quantile(0.75)
    iqr = q3 - q1
    low  = q1 - k*iqr; high = q3 + k*iqr
    is_out = (series < low) | (series > high)
    return is_out, low, high

WINDOW, K = 14, 2.5               # mini-exercise: change these two numbers
is_out, low, high = detect_outliers_iqr(y, window=WINDOW, k=K)

# Known-event calendar: the promo dates come from marketing, not from the data
promo_dates = pd.date_range('2022-10-28', periods=5, freq='D')
is_promo = pd.Series(y.index.isin(promo_dates), index=y.index, name='is_promo')

flagged = ', '.join(d.strftime('%Y-%m-%d') for d in y.index[is_out])
print(f'Statistical flags: {is_out.sum()} days ({is_out.mean():.1%} of days): {flagged}')
print(f'Promo days caught by the detector: {(is_out & is_promo).sum()} of {is_promo.sum()}')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y.index, y.values, color=NAVY, lw=0.7, label='Observed')
ax.fill_between(y.index, low, high, color=GOLD, alpha=0.2, label=f'IQR band (window={WINDOW}, k={K})')
ax.axvspan(promo_dates[0], promo_dates[-1] + pd.Timedelta(days=1),
           color=GREEN, alpha=0.25, label='Promo (calendar)')
ax.scatter(y.index[is_out], y.values[is_out], color=RED, s=18, zorder=5, label='Flagged by IQR rule')
ax.set_title('Outlier detection — rolling IQR rule vs known-event calendar',
             fontweight='bold', color=INK)
ax.legend(loc='upper left', fontsize=8); plt.tight_layout(); plt.show()

**Why did the detector miss the promo?** The 3 stockout days are flagged; the 5 promo days (+60 units each) are not — and they are not flagged even with a tighter band (`K = 1.5`). In a 14-day window, 5 of the 14 values are inflated, so the window's own Q3 and IQR rise with them: the band climbs with the spike (look at the plot: the gold band balloons around the promo). This is **masking** — a multi-day event hides itself by contaminating the statistics used to detect it.

Two lessons:
1. The window must be much longer than the events you want to catch (try it in the mini-exercise below).
2. For events you *already know about*, a calendar flag beats any detector. That is why `is_promo` exists: business knowledge first, statistics second.

In [ ]:
# Treatment depends on the business category, not on the detector alone
to_replace = is_out & ~is_promo      # data errors (here: the stockout) → replace
                                     # known events (is_promo) → keep in the series, keep the flag
# Rolling median of the clean neighbours (flagged days masked out first)
med7 = y.where(~is_out).rolling(7, center=True, min_periods=3).median()
y_clean = y.copy()
y_clean[to_replace] = med7[to_replace]
y_clean = y_clean.fillna(y.median())
print(f'Days replaced: {to_replace.sum()}   ·   Promo days kept and flagged: {is_promo.sum()}')
print(f'Mean before:  {y.mean():.2f}')
print(f'Mean after:   {y_clean.mean():.2f}  (small shift: only {to_replace.sum()} of {len(y)} days changed)')

> **Mini-exercise (parametric).** In the detector cell, change `WINDOW` from 14 to 7 and then to 28 (keep `K = 2.5`). Re-run and read the two printed lines. Then try `K = 1.5` with the window you prefer.
> - Which window catches the promo? Why does the short window miss *both* events?
> - What does `K = 1.5` add: real events or false alarms?
>
> Outlier handling is a **design choice**, not a single right answer — the trade-off is between losing real signal (over-cleaning) and contaminating the model (under-cleaning). Re-run the treatment cell too, and remember: anything downstream changes with this choice.
>
> One more trap: the detector runs on the whole series, holdout included. If it flags a day inside the holdout (section 6), the treatment changes the very actuals the models are scored against. In a real project, detect and clean on the training data only.

## 6 · Baseline forecasts

Before touching ARIMA or Prophet or a neural net, you owe these four baselines:

1. **Naive** — tomorrow = today
2. **Seasonal-Naive (SNaive)** — tomorrow = the *same weekday last week* (lag 7). In class we described SNaive as "last year, same week": the same idea with a yearly season instead of a weekly one.
3. **Linear trend** — extrapolate a regression line
4. **Holt-Winters (ETS)** — exponential smoothing with trend & seasonality

If a fancy model cannot beat these four, it has no business being in production.

In [ ]:
# Train/test split: last 60 days as holdout
H = 60
train, test = y_clean.iloc[:-H], y_clean.iloc[-H:]
print(f'Train: {len(train)} days   Test: {len(test)} days')
print(f'Test window: {test.index.min().date()} → {test.index.max().date()}')

In [ ]:
# Forecasts
fc = pd.DataFrame(index=test.index)
# Naive
fc['Naive']  = train.iloc[-1]
# Seasonal-Naive (weekly): holdout day h copies the same weekday in the last training week
fc['SNaive'] = train.iloc[-7 + (np.arange(H) % 7)].values
# Linear trend
t_tr = np.arange(len(train)); t_te = np.arange(len(train), len(train)+H)
slope, intercept = np.polyfit(t_tr, train.values, 1)
fc['LinearTrend'] = slope*t_te + intercept
# Holt-Winters
hw = ExponentialSmoothing(train, trend='add', seasonal='add',
                          seasonal_periods=7).fit()
fc['Holt-Winters'] = hw.forecast(H).values

src = train.index[-7 + (np.arange(7) % 7)]
print('SNaive weekday check:', '  '.join(f'{d:%a}←{s:%a}' for d, s in zip(test.index[:7], src)))
print(f"Holt-Winters smoothing: alpha={hw.params['smoothing_level']:.2f}  "
      f"beta={hw.params['smoothing_trend']:.2f}  gamma={hw.params['smoothing_seasonal']:.2f}")
fc.head()

In [ ]:
# Plot forecasts vs actuals on the holdout
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train.index[-90:], train.values[-90:], color=MUTED, lw=0.8, label='Train (last 90d)')
ax.plot(test.index, test.values, color=NAVY, lw=1.5, label='Actual')
for col, color in zip(['Naive','SNaive','LinearTrend','Holt-Winters'],
                      [RED, GOLD, GREEN, '#7B2CBF']):
    ax.plot(fc.index, fc[col], color=color, lw=1.0, ls='--', label=col)
ax.set_title(f'Holdout forecasts — last {H} days',
             fontweight='bold', color=INK)
ax.legend(loc='upper left', ncol=3, fontsize=8)
plt.tight_layout(); plt.show()

## 7 · Accuracy metrics — MAPE / WAPE / MASE + bias

Three classic forecasting metrics with very different sensitivities, plus a direction check:

- **MAPE** = mean of |error| / |actual|. Easy to communicate. **Explodes** when actuals are near zero (intermittent demand → unusable).
- **WAPE** = sum |error| / sum |actual|. The volume-weighted version. **Robust to zeros**. Industry standard for retail. On the slides it is called **WMAPE** — same metric.
- **MASE** = mean |error| on the holdout / mean |y(t) − y(t−7)| on the **training** set, i.e. the error a one-step seasonal-naive forecast makes *inside the training data*. **Scale-free**, so you can compare SKUs of very different size. MASE < 1 means the model beats that in-sample seasonal-naive error.
- **Bias** = mean(forecast − actual). Positive = over-forecasting (excess stock), negative = under-forecasting (lost sales). Not an accuracy metric but a direction check (the "Bias & Coverage" box on the accuracy-metrics slide).

In [ ]:
def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

def mase(y_true, y_pred, y_train, season=7):
    # scale = in-sample MAE of the seasonal-naive forecast: |y(t) - y(t-season)| on the training set
    naive_err = np.mean(np.abs(y_train[season:] - y_train[:-season]))
    return np.mean(np.abs(y_true - y_pred)) / naive_err

def bias(y_true, y_pred):
    return np.mean(y_pred - y_true)      # > 0 = over-forecast

metrics = pd.DataFrame(
    {col: {
        'MAPE': mape(test.values, fc[col].values),
        'WAPE': wape(test.values, fc[col].values),
        'MASE': mase(test.values, fc[col].values, train.values),
        'Bias': bias(test.values, fc[col].values),
    } for col in fc.columns}
).round(4)
metrics

**Read-outs** (H = 60, holdout 2 Nov – 31 Dec 2023).
- **Holt-Winters wins, but only just.** WAPE about 7.0% against about 7.8% for both Naive and SNaive: less than one point. Simple baselines hold their own — the "start simple, climb only when justified" rule made concrete. Linear trend is far behind (about 24%): a straight line cannot follow a seasonal decline.
- **Every MASE is above 1, SNaive included — and that is normal.** The MASE scale is the error of a seasonal-naive forecast made *one week ahead, inside the training data*. Here we forecast up to 60 days ahead, in Nov–Dec, when demand is falling. Errors grow with the horizon, so MASE > 1 at a long horizon is expected, not a failure. Use MASE to rank models and compare SKUs (lower is better), not as a pass/fail line at 1. With a 30-day holdout, Holt-Winters drops below 1 (try it below).
- **Bias:** every method over-forecasts on average — Naive only marginally (about +0.6 units a day), Holt-Winters by about 5. Section 8 explains why.
- WAPE is the cleanest cross-method comparison; communicate it to business.
- If a method wins on WAPE but loses on MAPE, its errors sit on low-volume days (MAPE over-weights them). Check whether those days matter to the business before trusting MAPE.

> **Mini-exercise (parametric).** Change `H = 60` to `H = 30` (shorter holdout, less generalisation evidence) and to `H = 120` (longer holdout), then run all cells from the train/test split onwards. Watch how the ranking of models can change. Forecast quality is *not* a constant; it depends on horizon and window.

## 8 · Residual diagnostics — what does the best model still miss?

If residuals look like white noise, you have extracted the signal. If they show structure, there is *more model* to build.

Two checks: the **holdout errors** of Holt-Winters (is the forecast biased? do the errors drift together?) and the **in-sample residuals** (Ljung-Box test: are they white noise?).

In [ ]:
resid = test.values - fc['Holt-Winters'].values     # holdout errors: actual − forecast
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(test.index, resid, color=NAVY, lw=0.9)
axes[0].axhline(0, color=MUTED, ls='--', lw=0.7)
axes[0].set_title('Holt-Winters holdout errors over time',
                   fontweight='bold', color=INK, fontsize=10)

axes[1].hist(resid, bins=20, color=NAVY, alpha=0.7, edgecolor='white')
axes[1].axvline(0, color=GOLD, ls='--', lw=0.8)
axes[1].set_title('Holdout error distribution',
                   fontweight='bold', color=INK, fontsize=10)

acf_lags = min(14, len(resid) - 1)                   # a short holdout allows fewer lags
plot_acf(resid, lags=acf_lags, ax=axes[2])
axes[2].set_title(f'Holdout error ACF (lags 1-{acf_lags})',
                   fontweight='bold', color=INK, fontsize=10)
plt.tight_layout(); plt.show()
print(f'Holdout error mean: {resid.mean():.2f}  (actual − forecast: ~0 if unbiased, negative = over-forecast)')
print(f'Holdout error std:  {resid.std():.2f}')
r_res = acf(resid, nlags=3)
print(f'Holdout error ACF, lags 1 / 2 / 3: {r_res[1]:.2f} / {r_res[2]:.2f} / {r_res[3]:.2f}')
lb = acorr_ljungbox(hw.resid, lags=[7, 14])
fmt_p = lambda p: '< 0.0001' if p < 1e-4 else f'= {p:.4f}'
print(f'Ljung-Box on in-sample residuals: p {fmt_p(lb.lb_pvalue.iloc[0])} (lag 7), '
      f'p {fmt_p(lb.lb_pvalue.iloc[1])} (lag 14)  → p < 0.05 = not white noise')

**Read-out.** The holdout errors average about −5 units a day: Holt-Winters over-forecasts by roughly 5 units every day. The cause is in the data, not in the code. The model knows the weekly cycle (period 7) but not the yearly one, and the holdout falls in Nov–Dec, when the yearly wave is going down (look back at the monthly profile in section 1). So the errors drift together (high ACF at short lags), and the Ljung-Box test on the in-sample residuals rejects white noise: there is *more model* to build.

Look also at the fitted smoothing parameters printed with the forecasts: beta and gamma come out at 0, so the model keeps a fixed slope and a weekly pattern that never updates — the α, β, γ of the Holt-Winters slide, read on real output.

**Next step:** add the yearly seasonality (MSTL, Fourier terms, a month regressor) or exogenous drivers (section 9).

## 9 · Caveats & next steps

1. **Synthetic data flatters models.** Real demand series have promotions, holidays, supply disruptions, price changes — exogenous regressors none of the four baselines use.
2. **Single point forecast hides uncertainty.** Always produce **prediction intervals**. For Holt-Winters, `hw.simulate(H, repetitions=500)` returns 500 simulated future paths: their 5th and 95th percentiles give a 90% interval. Then check the **coverage** — how often the actuals fall inside.
3. **Recent observations matter more.** Re-fit weekly in production; do not freeze a model from 6 months ago.
4. **Pick the metric for the audience.** WAPE for totals and business dashboards, MASE for cross-series comparison; MAPE only when the series has no zeros and the audience is non-technical.
5. **One holdout is one opinion.** Walk-forward validation (several forecast origins, each scored the same way) gives a more reliable ranking.

### Where to go next

- Add exogenous regressors: price, promotion calendar, weather → SARIMAX or Prophet
- Model weekly *and* yearly seasonality together: MSTL, Fourier terms
- Move to **probabilistic forecasts**: quantile regression, conformal intervals
- For the multi-product case (hierarchy, mix, reconciliation) → notebook **03_demand_eda_multi_product**

### References

- Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* (3rd ed.) — open access at otexts.com/fpp3
- Makridakis, Spiliotis, Assimakopoulos (2018). *Statistical and ML forecasting methods: concerns and ways forward*. PLoS One.
- Course deck, Module 5 *Sales Forecasting*: slides "Forecasting Philosophy", "Time Series Decomposition", "Holt-Winters Exponential Smoothing", "Forecast Accuracy Metrics", "Ensemble & Model Selection".

---

**End of notebook.** About 30 seconds of runtime on a laptop CPU.